# SpecX — pipeline walkthrough

This notebook runs each stage of the pipeline on a service manual so you can inspect intermediate output:

1. **Parse** — tables first, prose outside tables
2. **Clean + structure** — learned boilerplate, headings
3. **Chunk** — table rows and heading-bounded prose
4. **Index + retrieve** — BGE + BM25 → RRF → cross-encoder
5. **Extract** — LLM → Pydantic → grounding check → JSON / CSV

Put the manual at `data/f150-workshop-manual.pdf` (or change `PDF` below) and set `OPENROUTER_API_KEY` in `.env`.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
ROOT = os.path.abspath('..')
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'src'))
PDF = 'data/f150-workshop-manual.pdf'

## 1. Parse one page
Page 636 is the *Front Disc Brake* specifications page: three tables (materials, general specs, torque).

In [2]:
import pymupdf
from specx.ingest.parse import parse_page

doc = pymupdf.open(PDF)
page = parse_page(doc[635])
for t in page.tables:
    print('TABLE', t.title)
    for row in t.rows[:5]:
        print('   ', row)
print('\nPROSE LINES (outside tables):')
for l in page.lines[:8]:
    print(f'   y={l.y0:5.0f} size={l.size} bold={l.bold} | {l.text}')

TABLE Material
    ['Item', 'Specification', 'Fill Capacity']
    ['High Performance DOT 3 Motor Vehicle Brake Fluid PM-1-C (US); CPM-1-C (Canada)', 'WSS-M6C62-A or WSS-M6C65-A1', '1,420 ml (3 pt)']
    ['Metal Brake Parts Cleaner PM-4-A or PM-4-B (US); CPM-4 (Canada)', '—', '—']
TABLE General Specifications
    ['Item', 'Specification']
    ['Brake Disc', '']
    ['Brake disc minimum thickness', '32 mm (1.259 in)']
    ['Brake Pads', '']
    ['Brake pad maximum taper wear (in any direction)', '3.0 mm (0.118 in)']
TABLE Torque Specifications
    ['Description', 'Nm', 'lb-ft', 'lb-in']
    ['Brake caliper anchor plate bolts', '250', '184', '—']
    ['Brake caliper flow bolt', '35', '26', '—']
    ['Brake caliper guide pin bolts', '37', '27', '—']
    ['Brake disc shield bolts', '18', '—', '159']

PROSE LINES (outside tables):
   y=   59 size=11.0 bold=False | 2014 F-150 Workshop Manual
   y=   59 size=11.0 bold=False | Page 1 sur 2
   y=  122 size=8.2 bold=False | SECTION 206-03: Front 

## 2–3. Clean, structure and chunk the whole manual
Boilerplate is learned from the document itself: repeated headers/footers, TOC/index pages, figure codes.

In [3]:
from collections import Counter
from specx.ingest.chunk import build_chunks

chunks, stats = build_chunks(PDF, 'demo')
print(stats)
print(Counter(c.chunk_type for c in chunks), '| with values:', sum(c.has_spec for c in chunks))

for c in chunks:
    if c.page == 636 and c.chunk_type == 'table':
        print('\n---', c.heading_path)
        print(c.text)

BuildStats(pages=852, toc_pages=0, image_pages=0, blank_pages=0, ocr_pages=0, has_outline=False)
Counter({'text': 957, 'table': 649}) | with values: 904

--- SECTION 206-03: Front Disc Brake › Torque Specifications › Material
Item: High Performance DOT 3 Motor Vehicle Brake Fluid PM-1-C (US); CPM-1-C (Canada) | Specification: WSS-M6C62-A or WSS-M6C65-A1 | Fill Capacity: 1,420 ml (3 pt)
Item: Metal Brake Parts Cleaner PM-4-A or PM-4-B (US); CPM-4 (Canada) | Specification: — | Fill Capacity: —

--- SECTION 206-03: Front Disc Brake › Torque Specifications › General Specifications
Brake Disc › Item: Brake disc minimum thickness | Specification: 32 mm (1.259 in)
Brake Pads › Item: Brake pad maximum taper wear (in any direction) | Specification: 3.0 mm (0.118 in)
Brake Pads › Item: Brake pad minimum thickness | Specification: 3.0 mm (0.118 in)

--- SECTION 206-03: Front Disc Brake › Torque Specifications
Description: Brake caliper anchor plate bolts | Nm: 250 | lb-ft: 184 | lb-in: —
Descript

In [4]:
# A prose chunk: numbered steps stay together with their torque values
for c in chunks:
    if c.chunk_type == 'text' and 'Tighten the nuts to 55 Nm' in c.text:
        print(c.heading_path)
        print(c.text)
        break

SECTION 204-00: Suspension System — General Information › Camber and Caster Adjustment › Front of Lower Arm Rear of Lower Arm
9. With the weight of the vehicle resting on the wheel and tire assemblies, hold the front cam bolt and tighten the nut to 350 Nm (258 lb-ft).
10. With the weight of the vehicle resting on the wheel and tire assemblies, hold the rear cam bolt and tighten the nut to 350 Nm (258 lb-ft).
11. Install the 2 stabilizer bar bracket bolts and plate assembly.
12. Position the stabilizer bar, install the 2 stabilizer bar brackets and the 4 new stabilizer bar bracket nuts.
• Tighten the nuts to 55 Nm (41 lb-ft).
13. NOTICE: Do not hold the stabilizer link boot with any tool as damage to the boot will occur.
NOTE: Use the hex-holding feature to prevent the stud from turning while installing the nuts.
Install the 2 new stabilizer bar link lower nuts.
• Tighten the nuts to 80 Nm (59 lb-ft).
14. Check and, if necessary, adjust the front toe. For additional information, refer t

## 4. Index and retrieve
`ingest()` runs the stages above, embeds the chunks and persists FAISS + chunks (cached by file hash).

In [5]:
from specx import pipeline
from specx.query.retrieve import retrieve

info = pipeline.ingest(PDF)
index = pipeline.load_index(info.doc_id)

q = 'Torque for brake caliper bolts'
for h in retrieve(index, q):
    print(f'{h.score:6.2f}  D={h.dense_rank}  S={h.sparse_rank}  p.{h.chunk.page:<4} {h.chunk.chunk_type:<5} {h.chunk.text[:90]!r}')

  8.30  D=1  S=0  p.636  table 'Description: Brake caliper anchor plate bolts | Nm: 250 | lb-ft: 184 | lb-in: —\nDescriptio'
  8.10  D=0  S=1  p.652  table 'Description: Brake caliper guide pin bolts | Nm: 33 | lb-ft: 24 | lb-in: —\nDescription: Br'
  6.62  D=2  S=2  p.652  table 'Description: Axle vent valve | Nm: 18 | lb-ft: — | lb-in: 159\nDescription: Brake caliper b'
  5.83  D=4  S=4  p.599  table 'Description: Brake caliper bleed screw | Nm: 10 | lb-ft: — | lb-in: 89\nDescription: Master'
  4.61  D=7  S=None  p.636  table 'Description: Brake disc shield bolts | Nm: 18 | lb-ft: — | lb-in: 159\nDescription: Brake f'
  4.10  D=11  S=17  p.672  table 'Description: Parking brake cable bracket bolt (RH) | Nm: 30 | lb-ft: 22 | lb-in: —\nDescrip'
  3.77  D=10  S=None  p.59   table 'Description: Brake disc shield bolts | Nm: 17 | lb-ft: — | lb-in: 150\nDescription: Brake h'
  3.76  D=9  S=None  p.24   table 'Description: Brake disc shield bolts | Nm: 17 | lb-ft: — | lb-in: 150\nDescriptio

## 5. Extract structured specs

In [6]:
result = pipeline.query(info.doc_id, q)
print('mode:', result.mode, '| cached:', result.cached, '| timings (ms):', result.timings_ms, '| rejected:', len(result.rejected))
print(json.dumps([s.required_fields() for s in result.specs], indent=2, ensure_ascii=False))

mode: llm | cached: True | timings (ms): {'retrieve': 35.5, 'rerank': 1224.1, 'extract': 6107.2} | rejected: 0
[
  {
    "component": "Brake caliper anchor plate bolts",
    "spec_type": "Torque",
    "value": "250",
    "unit": "Nm"
  },
  {
    "component": "Brake caliper flow bolt",
    "spec_type": "Torque",
    "value": "35",
    "unit": "Nm"
  },
  {
    "component": "Brake caliper guide pin bolts",
    "spec_type": "Torque",
    "value": "37",
    "unit": "Nm"
  },
  {
    "component": "Brake caliper guide pin bolts",
    "spec_type": "Torque",
    "value": "33",
    "unit": "Nm"
  },
  {
    "component": "Brake caliper support bracket bolts",
    "spec_type": "Torque",
    "value": "150",
    "unit": "Nm"
  }
]


In [7]:
# Full records with provenance
for s in result.specs:
    print(f'{s.component:<40} {s.value:>6} {s.unit or "":<4} ({s.alt_value} {s.alt_unit})  p.{s.page}  conf={s.confidence}  quote={s.source_quote!r}')

Brake caliper anchor plate bolts            250 Nm   (184 lb-ft)  p.636  conf=1.0  quote='Brake caliper anchor plate bolts | Nm: 250 | lb-ft: 184'
Brake caliper flow bolt                      35 Nm   (26 lb-ft)  p.636  conf=1.0  quote='Brake caliper flow bolt | Nm: 35 | lb-ft: 26'
Brake caliper guide pin bolts                37 Nm   (27 lb-ft)  p.636  conf=1.0  quote='Brake caliper guide pin bolts | Nm: 37 | lb-ft: 27'
Brake caliper guide pin bolts                33 Nm   (24 lb-ft)  p.652  conf=1.0  quote='Brake caliper guide pin bolts | Nm: 33 | lb-ft: 24'
Brake caliper support bracket bolts         150 Nm   (111 lb-ft)  p.652  conf=1.0  quote='Brake caliper support bracket bolts | Nm: 150 | lb-ft: 111'


In [8]:
# A few more spec types
for q in ['front brake fluid capacity', 'minimum brake disc thickness', 'stabilizer bar link nut torque']:
    r = pipeline.query(info.doc_id, q)
    print('\n#', q)
    for s in r.specs:
        print('   ', json.dumps(s.required_fields(), ensure_ascii=False), f'p.{s.page}')


# front brake fluid capacity
    {"component": "High Performance DOT 3 Motor Vehicle Brake Fluid", "spec_type": "Capacity", "value": "1,420", "unit": "ml"} p.636
    {"component": "High Performance DOT 3 Motor Vehicle Brake Fluid", "spec_type": "Capacity", "value": "740", "unit": "ml"} p.747
    {"component": "Motorcraft® High Performance DOT 3 Motor Vehicle Brake Fluid (US) / Motorcraft® Brake Fluid - Heavy Duty DOT 3 (Canada)", "spec_type": "Capacity", "value": "740", "unit": "ml"} p.696
    {"component": "Motorcraft® High Performance DOT 3 Motor Vehicle Brake Fluid (US) / Motorcraft® Brake Fluid - Heavy Duty DOT 3 (Canada)", "spec_type": "Capacity", "value": "1,420", "unit": "ml"} p.599

# minimum brake disc thickness
    {"component": "Front brake disc (minimum)", "spec_type": "Thickness", "value": "32", "unit": "mm"} p.599
    {"component": "Rear brake disc (minimum)", "spec_type": "Thickness", "value": "23.0", "unit": "mm"} p.599
    {"component": "Brake disc (minimum)", "spec_t

In [9]:
# CSV export
import csv, io
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=['component', 'spec_type', 'value', 'unit'])
w.writeheader()
w.writerows(s.required_fields() for s in result.specs)
print(buf.getvalue())

component,spec_type,value,unit
Brake caliper anchor plate bolts,Torque,250,Nm
Brake caliper flow bolt,Torque,35,Nm
Brake caliper guide pin bolts,Torque,37,Nm
Brake caliper guide pin bolts,Torque,33,Nm
Brake caliper support bracket bolts,Torque,150,Nm

